# ModernFinBERT v1 vs v2: Benchmark Inference

Side-by-side evaluation on held-out benchmarks:
1. Financial PhraseBank (50agree & allagree) — sentence-level
2. FiQA 2018 — sentence-level
3. Twitter Financial News Sentiment — sentence-level
4. FinEntity (SEntFiN) — entity-level sentiment

Models compared:
- `neoyipeng/ModernFinBERT-base` (v1, paper model — single-sentence inputs)
- `neoyipeng/ModernFinBERT-v2-medium` (v2, entity-aware — sentence-pair inputs `[CLS] entity [SEP] text [SEP]`)

For benchmarks without entities (FPB, FiQA, Twitter) v2 receives an empty entity (consistent with its training when no entity is known).
For FinEntity v2 uses the gold entity in segment A; v1 uses the legacy `f"{entity}: {text}"` prefix in single-sentence form.

In [ ]:
!pip install -q transformers datasets peft accelerate scikit-learn 2>/dev/null

In [ ]:
import torch
import numpy as np
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
MAX_LENGTH = 1024

MODELS = [
    {"label": "v1-base",   "hf_id": "neoyipeng/ModernFinBERT-base",      "entity_aware": False},
    {"label": "v2-medium", "hf_id": "neoyipeng/ModernFinBERT-v2-medium", "entity_aware": True},
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Models to compare: {[m['label'] for m in MODELS]}")

In [ ]:
def load_model(hf_id):
    tok = AutoTokenizer.from_pretrained(hf_id)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=NUM_CLASSES, attn_implementation="eager",
    ).to(device).eval()
    return tok, mdl

In [ ]:
def run_inference(tokenizer, model, texts, entities=None, batch_size=32):
    """If entities is provided (same length as texts), use sentence-pair tokenization."""
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            if entities is None:
                inputs = tokenizer(
                    batch_texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LENGTH,
                )
            else:
                batch_entities = entities[i : i + batch_size]
                inputs = tokenizer(
                    batch_entities, batch_texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LENGTH,
                )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def evaluate_benchmark(tokenizer, model, texts, labels, name, entities=None):
    preds = run_inference(tokenizer, model, texts, entities=entities)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    report = classification_report(
        labels, preds, target_names=LABEL_NAMES,
        output_dict=True, zero_division=0,
    )
    per_class = {c: round(report[c]["f1-score"], 4) for c in LABEL_NAMES}
    print(f"  {name}: acc={acc:.4f}, macro_f1={f1:.4f}, per_class={per_class}")
    return {"accuracy": round(acc, 4), "macro_f1": round(f1, 4), "per_class_f1": per_class, "n": len(texts)}


def empty_entities(n):
    return [""] * n

## 1. Financial PhraseBank (sentence-level)

In [ ]:
fpb50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb50)}, AllAgree: {len(fpb_all)}")

all_results = {m["label"]: {} for m in MODELS}

for m in MODELS:
    print(f"\n{'='*60}\nMODEL: {m['label']} ({m['hf_id']})\n{'='*60}")
    tok, mdl = load_model(m["hf_id"])
    ents50 = empty_entities(len(fpb50["sentence"])) if m["entity_aware"] else None
    entsA  = empty_entities(len(fpb_all["sentence"])) if m["entity_aware"] else None

    print("\n[FPB 50agree]")
    all_results[m["label"]]["fpb_50agree"] = evaluate_benchmark(
        tok, mdl, fpb50["sentence"], fpb50["label"], "FPB 50agree", entities=ents50,
    )
    print("\n[FPB AllAgree]")
    all_results[m["label"]]["fpb_allagree"] = evaluate_benchmark(
        tok, mdl, fpb_all["sentence"], fpb_all["label"], "FPB AllAgree", entities=entsA,
    )
    del tok, mdl
    if device.type == "cuda":
        torch.cuda.empty_cache()

## 2. FiQA 2018 (sentence-level, discretized)

In [ ]:
fiqa = load_dataset("pauri32/fiqa-2018", split="train")
fiqa_texts, fiqa_labels = [], []
LO, HI = -0.2, 0.2
for row in fiqa:
    fiqa_texts.append(row["sentence"])
    score = row["sentiment_score"]
    if score < LO:
        fiqa_labels.append(0)
    elif score > HI:
        fiqa_labels.append(2)
    else:
        fiqa_labels.append(1)
print(f"FiQA: {len(fiqa_texts)} rows")

for m in MODELS:
    print(f"\n[{m['label']}] FiQA 2018")
    tok, mdl = load_model(m["hf_id"])
    ents = empty_entities(len(fiqa_texts)) if m["entity_aware"] else None
    all_results[m["label"]]["fiqa_2018"] = evaluate_benchmark(
        tok, mdl, fiqa_texts, fiqa_labels, "FiQA 2018", entities=ents,
    )
    del tok, mdl
    if device.type == "cuda":
        torch.cuda.empty_cache()

## 3. Twitter Financial News Sentiment (sentence-level)

In [ ]:
tfns = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
tfns_remap = {0: 0, 1: 2, 2: 1}  # Bearish->NEG, Bullish->POS, Neutral->NEU
tfns_labels = [tfns_remap[l] for l in tfns["label"]]
print(f"Twitter Financial: {len(tfns_labels)} rows")

for m in MODELS:
    print(f"\n[{m['label']}] Twitter Financial News")
    tok, mdl = load_model(m["hf_id"])
    ents = empty_entities(len(tfns["text"])) if m["entity_aware"] else None
    all_results[m["label"]]["twitter_financial"] = evaluate_benchmark(
        tok, mdl, tfns["text"], tfns_labels, "Twitter Financial", entities=ents,
    )
    del tok, mdl
    if device.type == "cuda":
        torch.cuda.empty_cache()

## 4. FinEntity / SEntFiN (entity-level sentiment)

Each row has a sentence with multiple entity annotations, each with its own sentiment.
We evaluate per-entity: for each entity span, we feed the full sentence to the model
and compare the model's sentence-level prediction against the entity's gold sentiment.
This tests whether the model captures the dominant entity's sentiment correctly.

In [ ]:
finentity = load_dataset("yixuantt/FinEntity", split="train")
print(f"FinEntity: {len(finentity)} sentences")

ENTITY_LABEL_MAP = {"Positive": 2, "Negative": 0, "Neutral": 1}

# Sentence-level (majority entity sentiment) — no entity attached
sent_texts, sent_labels = [], []
for row in finentity:
    anns = row["annotations"]
    if not anns:
        continue
    counts = {}
    for ann in anns:
        l = ENTITY_LABEL_MAP.get(ann["label"], 1)
        counts[l] = counts.get(l, 0) + 1
    sent_texts.append(row["content"])
    sent_labels.append(max(counts, key=counts.get))

# Per-entity — both models get the entity in their preferred form
entity_names, entity_texts_v1, entity_labels = [], [], []
raw_texts_per_entity = []
for row in finentity:
    for ann in row["annotations"]:
        entity_names.append(ann["value"])
        entity_texts_v1.append(f"{ann['value']}: {row['content']}")
        entity_labels.append(ENTITY_LABEL_MAP.get(ann["label"], 1))
        raw_texts_per_entity.append(row["content"])

print(f"Sentence-level: {len(sent_texts)} sentences")
print(f"Per-entity: {len(entity_texts_v1)} pairs (NEG={entity_labels.count(0)}, NEU={entity_labels.count(1)}, POS={entity_labels.count(2)})")

# Conflicting-entity subset
conflict_names, conflict_texts_v1, conflict_labels, conflict_raw = [], [], [], []
for row in finentity:
    anns = row["annotations"]
    if len(anns) < 2 or len(set(a["label"] for a in anns)) < 2:
        continue
    for ann in anns:
        conflict_names.append(ann["value"])
        conflict_texts_v1.append(f"{ann['value']}: {row['content']}")
        conflict_labels.append(ENTITY_LABEL_MAP.get(ann["label"], 1))
        conflict_raw.append(row["content"])

print(f"Conflicting-entity pairs: {len(conflict_texts_v1)}")

for m in MODELS:
    print(f"\n[{m['label']}] FinEntity (3 views)")
    tok, mdl = load_model(m["hf_id"])

    ents = empty_entities(len(sent_texts)) if m["entity_aware"] else None
    all_results[m["label"]]["finentity_sentence"] = evaluate_benchmark(
        tok, mdl, sent_texts, sent_labels, "FinEntity sentence-level", entities=ents,
    )

    if m["entity_aware"]:
        all_results[m["label"]]["finentity_per_entity"] = evaluate_benchmark(
            tok, mdl, raw_texts_per_entity, entity_labels, "FinEntity per-entity",
            entities=entity_names,
        )
    else:
        all_results[m["label"]]["finentity_per_entity"] = evaluate_benchmark(
            tok, mdl, entity_texts_v1, entity_labels, "FinEntity per-entity",
        )

    if conflict_texts_v1:
        if m["entity_aware"]:
            all_results[m["label"]]["finentity_conflict"] = evaluate_benchmark(
                tok, mdl, conflict_raw, conflict_labels, "FinEntity conflicting",
                entities=conflict_names,
            )
        else:
            all_results[m["label"]]["finentity_conflict"] = evaluate_benchmark(
                tok, mdl, conflict_texts_v1, conflict_labels, "FinEntity conflicting",
            )
    else:
        all_results[m["label"]]["finentity_conflict"] = {"accuracy": 0, "macro_f1": 0, "n": 0}

    del tok, mdl
    if device.type == "cuda":
        torch.cuda.empty_cache()

## Results Summary

In [ ]:
BENCHMARKS = [
    "fpb_50agree", "fpb_allagree", "fiqa_2018",
    "twitter_financial", "finentity_sentence",
    "finentity_per_entity", "finentity_conflict",
]

labels = [m["label"] for m in MODELS]

print("\n" + "="*90)
print("RESULTS SUMMARY — ModernFinBERT v1 vs v2")
print("="*90)
header = f"{'Benchmark':<28} {'N':>6}"
for lbl in labels:
    header += f"   {lbl+' acc':>12} {lbl+' F1':>12}"
print(header)
print("-"*90)
for bench in BENCHMARKS:
    n = next((all_results[lbl][bench]["n"] for lbl in labels if bench in all_results[lbl]), "-")
    row = f"{bench:<28} {str(n):>6}"
    for lbl in labels:
        r = all_results[lbl].get(bench, {})
        row += f"   {r.get('accuracy', 0):>12.4f} {r.get('macro_f1', 0):>12.4f}"
    print(row)
print()
print("Δ (v2-medium − v1-base):")
print(f"  {'Benchmark':<28} {'Δ acc':>10} {'Δ F1':>10}")
for bench in BENCHMARKS:
    r1 = all_results.get("v1-base", {}).get(bench, {})
    r2 = all_results.get("v2-medium", {}).get(bench, {})
    if not r1 or not r2:
        continue
    d_acc = r2.get("accuracy", 0) - r1.get("accuracy", 0)
    d_f1  = r2.get("macro_f1", 0) - r1.get("macro_f1", 0)
    print(f"  {bench:<28} {d_acc:>+10.4f} {d_f1:>+10.4f}")

with open("benchmark_results_v1_vs_v2.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved to benchmark_results_v1_vs_v2.json")